# Step 7 — Feature Extraction and Engineering

The selected temporal design uses clinical information recorded during
the first 2 hours following ICU admission to predict clinical
deterioration occurring after the 2-hour landmark and through 8 hours
after ICU admission.

Only information available at or before the prediction landmark is used
to construct predictor variables, thereby maintaining temporal
separation between predictors and outcomes.

Features are derived from routinely collected physiological
measurements and laboratory investigations available in MIMIC-IV Demo.

In [3]:
import pandas as pd
from pathlib import Path

# Resolve the project root from this notebook's location
project_root = Path.cwd().resolve()
if not (project_root / "data" / "icu").exists():
    project_root = project_root.parent

project_root = project_root.resolve()

data_dir = project_root / "data"
results_dir = project_root / "results"

# Load the full deterioration outcome table for the 2h -> 8h feature design
outcomes_path = results_dir / "deterioration_outcomes.csv"
if not outcomes_path.exists():
    outcomes_path = results_dir / "landmark_6h_outcomes.csv"
    if not outcomes_path.exists():
        raise FileNotFoundError(
            "Outcome CSV not found in results/. Expected either "
            "deterioration_outcomes.csv or landmark_6h_outcomes.csv"
        )

outcomes = pd.read_csv(outcomes_path)

# Resolve the chart-events file path
expected_path = data_dir / "icu" / "chartevents.csv"
search_roots = [
    project_root,
    data_dir,
    data_dir / "icu",
]
path_candidates = [
    "icu/chartevents.csv",
    "chartevents.csv",
    "icu/chartevents.csv.gz",
    "chartevents.csv.gz",
]

candidate_paths = [expected_path]
for root in search_roots:
    for candidate in path_candidates:
        candidate_paths.append(Path(root) / candidate)

candidate_paths = list(dict.fromkeys(candidate_paths))
matches = [path for path in candidate_paths if path.is_file()]

if not matches:
    raise FileNotFoundError(
        "chartevents.csv was not found. Expected location:\n"
        f"{expected_path}"
    )

chartevents_path = matches[0]
print(f"Using chart events file: {chartevents_path}")

# Ensure outcome timestamps are datetime values
datetime_columns = [
    "intime",
    "outtime",
    "deathtime",
    "death_event_time",
    "ventilation_time",
    "vasopressor_time",
    "landmark_time",
    "prediction_end",
]

for column in datetime_columns:
    if column in outcomes.columns:
        outcomes[column] = pd.to_datetime(
            outcomes[column],
            errors="coerce"
        )

# Validate the outcome table
required_columns = {
    "subject_id",
    "hadm_id",
    "stay_id",
    "intime",
    "landmark_time",
    "prediction_end",
    "eligible_at_landmark",
    "deterioration_6_12h",
}

missing_columns = required_columns.difference(outcomes.columns)

if missing_columns:
    raise ValueError(
        f"Missing required outcome columns: {sorted(missing_columns)}"
    )

print(f"Validated outcomes: {outcomes.shape}")

# Required summary for the 2h -> 8h feature design
summary_df = outcomes.copy()
summary_df["landmark_time"] = summary_df["intime"] + pd.Timedelta(hours=2)
summary_df["prediction_end_time"] = summary_df["intime"] + pd.Timedelta(hours=8)
summary_df["reaches_landmark"] = summary_df["outtime"] >= summary_df["landmark_time"]
# Normalize timestamps consistently before comparisons
for column in ["event_time", "landmark_time"]:
    summary_df[column] = pd.to_datetime(
        summary_df[column],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

summary_df["event_before_or_at_landmark"] = (
    summary_df["event_time"].notna()
    & summary_df["landmark_time"].notna()
    & (summary_df["event_time"] <= summary_df["landmark_time"])
)
summary_df["eligible"] = (
    summary_df["reaches_landmark"]
    & ~summary_df["event_before_or_at_landmark"]
)
summary_df["future_deterioration"] = (
    summary_df["eligible"]
    & summary_df["event_time"].notna()
    & (summary_df["event_time"] > summary_df["landmark_time"])
    & (summary_df["event_time"] <= summary_df["prediction_end_time"])
).astype(int)

eligible = summary_df.loc[summary_df["eligible"]].copy()

print("\nEligible ICU stays:", len(eligible))
print("Unique patients:", eligible["subject_id"].nunique())
print("\nOutcome:")
print(
    eligible["future_deterioration"]
    .value_counts()
    .sort_index()
    .to_string()
)

# Define vital-sign item IDs used for feature extraction.
# These are standard MIMIC-IV chartevents item IDs.
VITAL_ITEMIDS = {
    "heart_rate": 220045,
    "systolic_bp": 220179,
    "diastolic_bp": 220180,
    "mean_bp": 220181,
    "respiratory_rate": 220210,
    "temperature": 223761,
    "spo2": 220277,
}

# Define the cohort used for feature extraction.
cohort = eligible.copy()

print("=" * 70)
print("STEP 7 — FEATURE ENGINEERING")
print("=" * 70)

print("\nCohort:")
print("Eligible ICU stays:", len(cohort))
print("Unique patients:", cohort["subject_id"].nunique())
print("\nOutcome:")
print(cohort["future_deterioration"].value_counts().sort_index().to_string())

# Load chart events from CSV or CSV.GZ
chartevents = pd.read_csv(
    chartevents_path,
    compression="infer",
    usecols=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "charttime",
        "itemid",
        "valuenum",
    ],
    low_memory=False,
)

chartevents["charttime"] = pd.to_datetime(
    chartevents["charttime"],
    errors="coerce",
)

# Keep only vital-sign measurements from the eligible cohort
vitals = chartevents[
    chartevents["itemid"].isin(VITAL_ITEMIDS.values())
].copy()

vitals = vitals[
    vitals["stay_id"].isin(cohort["stay_id"])
].copy()

print("Loaded chart events:", chartevents.shape)
print("Vital-sign records:", vitals.shape)

itemid_to_vital = {
    itemid: variable
    for variable, itemid in VITAL_ITEMIDS.items()
}

vitals["variable"] = (
    vitals["itemid"].map(itemid_to_vital)
)

vitals = vitals.merge(
    cohort[
        [
            "subject_id",
            "stay_id",
            "intime",
            "landmark_time"
        ]
    ],
    on=["subject_id", "stay_id"],
    how="inner"
)

vitals_obs = vitals[
    (vitals["charttime"] >= vitals["intime"])
    &
    (vitals["charttime"] <= vitals["landmark_time"])
].copy()

print(
    "Vital measurements in observation window:",
    len(vitals_obs)
)

print(
    "ICU stays represented:",
    vitals_obs["stay_id"].nunique()
)

vitals_obs["variable"].value_counts()

print(
    "Vital measurements in observation window:",
    len(vitals_obs)
)

print(
    "ICU stays represented:",
    vitals_obs["stay_id"].nunique()
)

vitals_obs["variable"].value_counts()

vitals_obs = vitals_obs.sort_values(
    ["stay_id", "variable", "charttime"]
)

vital_summary = (
    vitals_obs
    .groupby(["stay_id", "variable"])
    .agg(
        mean=("valuenum", "mean"),
        min=("valuenum", "min"),
        max=("valuenum", "max"),
        std=("valuenum", "std"),
        last=("valuenum", "last"),
        count=("valuenum", "count")
    )
    .reset_index()
)

vital_summary.head()

vital_features = vital_summary.pivot(
    index="stay_id",
    columns="variable"
)

vital_features.columns = [
    f"{variable}_{statistic}"
    for statistic, variable in vital_features.columns
]

vital_features = (
    vital_features
    .reset_index()
)

vital_features.head()

model_data = cohort[
    [
        "subject_id",
        "hadm_id",
        "stay_id",
        "future_deterioration"
    ]
].merge(
    vital_features,
    on="stay_id",
    how="left"
)

print("Rows:", len(model_data))
print(
    "Unique stays:",
    model_data["stay_id"].nunique()
)

print("\nOutcome:")
print(
    model_data["future_deterioration"].value_counts()
)

model_data.head()

feature_columns = [
    col for col in model_data.columns
    if col not in [
        "subject_id",
        "hadm_id",
        "stay_id",
        "future_deterioration"
    ]
]

missingness = (
    model_data[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missingness

# Vital-sign coverage across the eligible cohort
cohort_stays = cohort["stay_id"].nunique()

vital_coverage = (
    vitals_obs
    .dropna(subset=["valuenum"])
    .groupby("variable")["stay_id"]
    .nunique()
    .reindex(list(VITAL_ITEMIDS.keys()), fill_value=0)
    .rename("stays_with_measurement")
    .to_frame()
)

vital_coverage["cohort_stays"] = cohort_stays
vital_coverage["coverage_percent"] = (
    vital_coverage["stays_with_measurement"]
    .div(cohort_stays)
    .mul(100)
    .round(2)
)

vital_coverage = vital_coverage.reset_index()
vital_coverage

# ============================================================
# FINAL STEP 7 DATASET VALIDATION
# ============================================================

forbidden_terms = [
    "event_time",
    "event_type",
    "deterioration_time",
    "hours_to_deterioration",
    "deathtime",
    "ventilation_time",
    "vasopressor_time",
    "prediction_end",
    "landmark_time"
]

potential_leakage = [
    col for col in model_data.columns
    if any(term in col.lower() for term in forbidden_terms)
]

feature_columns = [
    col for col in model_data.columns
    if col not in ["subject_id", "hadm_id", "stay_id", "future_deterioration"]
]

print("\n")
print("=" * 54)
print("FINAL STEP 7 DATASET")
print("=" * 54)
print("Rows:", len(model_data))
print("Unique patients:", model_data["subject_id"].nunique())
print("Positive outcomes:", int(model_data["future_deterioration"].sum()))
print("Negative outcomes:", int((model_data["future_deterioration"] == 0).sum()))
print("Total predictor columns:", len(feature_columns))
print("Potential leakage columns:", potential_leakage)
print("\n")
print("=" * 54)
print("STEP 7 COMPLETE")
print("=" * 54)

# Ensure the results directory exists
results_dir.mkdir(parents=True, exist_ok=True)

# Validate the dataset before saving
if "model_data" not in globals():
    raise RuntimeError("model_data has not been created.")

if model_data.empty:
    raise ValueError("model_data is empty; nothing to save.")

output_path = results_dir / "model_features_2h.csv"

# Save the feature dataset
model_data.to_csv(output_path, index=False)

print(f"Saved model features to: {output_path}")
print(f"Dataset shape: {model_data.shape}")

try:
    print("SPO2_ITEMID =", SPO2_ITEMID)
except NameError:
    print("SPO2_ITEMID variable not available in this notebook.")

    spo2_itemid = VITAL_ITEMIDS["spo2"]
    SPO2_ITEMID = spo2_itemid

    chartevents.loc[
        chartevents["itemid"].eq(spo2_itemid),
        ["itemid", "valuenum"]
    ].head()


Using chart events file: C:\Users\HARIKRISHNAN\Documents\Phoenix Code\default project\mimic-deterioration-ai\data\icu\chartevents.csv
Validated outcomes: (140, 25)

Eligible ICU stays: 99
Unique patients: 79

Outcome:
future_deterioration
0    81
1    18
STEP 7 — FEATURE ENGINEERING

Cohort:
Eligible ICU stays: 99
Unique patients: 79

Outcome:
future_deterioration
0    81
1    18
Loaded chart events: (668862, 6)
Vital-sign records: (38178, 6)
Vital measurements in observation window: 1628
ICU stays represented: 78
Vital measurements in observation window: 1628
ICU stays represented: 78
Rows: 99
Unique stays: 99

Outcome:
future_deterioration
0    81
1    18
Name: count, dtype: int64


FINAL STEP 7 DATASET
Rows: 99
Unique patients: 79
Positive outcomes: 18
Negative outcomes: 81
Total predictor columns: 42
Potential leakage columns: []


STEP 7 COMPLETE
Saved model features to: C:\Users\HARIKRISHNAN\Documents\Phoenix Code\default project\mimic-deterioration-ai\results\model_features_2h.c